In [1]:
import numpy as np
print(np.__version__)

from datetime import datetime
import torch
print(torch.__version__)

import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

from tqdm.notebook import trange

import random
import math
import time

1.26.4
2.9.0


In [42]:
import numpy as np
from numba import njit

SIZE = 5
EMPTY = 0
P1 = 1
P2 = -1
max_moves = 100


def build_action_map(size):
    actions = []

    for r in range(size):
        for c in range(size):
            if not (r == 0 or r == size-1 or c == 0 or c == size-1):
                continue

            if c != 0:
                actions.append((r, c, r, 0))
            if c != size - 1:
                actions.append((r, c, r, size - 1))
            if r != 0:
                actions.append((r, c, 0, c))
            if r != size - 1:
                actions.append((r, c, size - 1, c))

    return np.array(actions, dtype=np.int8)  # (A, 4)

@njit
def get_valid_moves_numba(state, action_map):
    A = action_map.shape[0]
    valid = np.zeros(A, dtype=np.uint8)

    for i in range(A):
        r, c = action_map[i, 0], action_map[i, 1]
        piece = state[r, c]

        if piece == EMPTY or piece == P1:
            valid[i] = 1

    return valid

@njit
def push_numba(state, action):
    r, c, dr, dc = action
    size = state.shape[0]

    new = state.copy()

    if r == dr:
        # row shift
        if dc == 0:
            # push from right → shift right
            for j in range(c, 0, -1):
                new[r, j] = new[r, j-1]
            new[r, 0] = P1
        else:
            # push from left → shift left
            for j in range(c, size-1):
                new[r, j] = new[r, j+1]
            new[r, size-1] = P1

    else:
        # column shift
        if dr == 0:
            for i in range(r, 0, -1):
                new[i, c] = new[i-1, c]
            new[0, c] = P1
        else:
            for i in range(r, size-1):
                new[i, c] = new[i+1, c]
            new[size-1, c] = P1

    return new

@njit
def has_line_numba(state, player):
    size = state.shape[0]

    for i in range(size):
        if np.all(state[i, :] == player):
            return True
        if np.all(state[:, i] == player):
            return True

    diag1 = True
    diag2 = True

    for i in range(size):
        if state[i, i] != player:
            diag1 = False
        if state[i, size - 1 - i] != player:
            diag2 = False

    return diag1 or diag2

def encode_state_batch(states):
    # states: (B, S, S)

    p2 = (states == P2)
    empty = (states == EMPTY)
    p1 = (states == P1)

    encoded = np.stack((p2, empty, p1), axis=1).astype(np.float32)
    # (B, 3, S, S)

    return encoded

class QuixoFast:
    def __init__(self):
        self.size = SIZE
        self.action_map = build_action_map(SIZE)   # (A, 4)
        self.action_size = self.action_map.shape[0]

        # --- NEW: fast lookup ---
        self._action_to_index = {}
        for i in range(self.action_size):
            key = tuple(self.action_map[i])
            self._action_to_index[key] = i

    def get_initial_state(self):
        return np.zeros((self.size, self.size), dtype=np.int8)

    def change_perspective(self, state, player):
        return state * player

    def get_valid_moves(self, state):
        return get_valid_moves_numba(state, self.action_map)

    def get_next_state(self, state, action, player):
        canonical = state * player
        next_state = push_numba(canonical, self.action_map[action])
        return next_state * player

    def get_value_and_terminated(self, state, player, move_count):
        if has_line_numba(state, -player):
            return -1, True
        if has_line_numba(state, player):
            return 1, True
        if move_count >= max_moves:
            return 0, True
        return 0, False
    
    def encode_action(self, r, c, dr, dc):
        return self._action_to_index[(r, c, dr, dc)]

    def decode_action(self, action):
        return self.action_map[action]
    
    def get_encoded_state(self, state):
        """
        Encodes the state into a 3-channel representation for the ResNet.
        Channels: [Opponent's Pieces, Empty Squares, Current Player's Pieces]
        """
        # Constants used: P1 = 1, P2 = -1, EMPTY = 0
        if state.ndim == 2: # Single state (5, 5)
            return np.stack((state == -1, state == 0, state == 1)).astype(np.float32)
        
        # Batch of states (Batch, 5, 5)
        # We swap axes so the output is (Batch, 3, 5, 5)
        encoded = np.stack((state == -1, state == 0, state == 1)).astype(np.float32)
        return np.swapaxes(encoded, 0, 1)

    def get_opponent(self, player):
        """Returns the other player (-1 for 1, 1 for -1)."""
        return -player

    def get_opponent_value(self, value):
        """Flips the value for the other player's perspective."""
        return -value

In [43]:
# check number of valid moves for the initital state
game = QuixoFast()
initial_state = game.get_initial_state()
valid_moves = game.get_valid_moves(initial_state)
print(f"Number of valid moves from the initial state: {valid_moves.sum()}")

Number of valid moves from the initial state: 44


In [44]:
def print_board(state):
    symbols = {0: ".", 1: "X", -1: "O"}
    print()
    for r in range(state.shape[0]):
        print(" ".join(symbols[x] for x in state[r]))
    print()


def get_human_move(game, state):
    """
    Input format: r c side (or 'q' to quit)
    """
    side_map = {
        "L": lambda r, c: (r, c, r, 0),
        "R": lambda r, c: (r, c, r, game.size - 1),
        "T": lambda r, c: (r, c, 0, c),
        "B": lambda r, c: (r, c, game.size - 1, c),
    }

    valid = game.get_valid_moves(state)

    while True:
        try:
            raw_input = input("Enter move (r c side[L/R/T/B]) or 'q' to quit: ").strip().lower()
            
            # Check for quit command
            if raw_input == 'q':
                return None

            raw = raw_input.split()
            r, c = int(raw[0]), int(raw[1])
            side = raw[2].upper()

            if side not in side_map:
                raise ValueError

            move = side_map[side](r, c)

            if move not in game.action_map:
                print("Illegal geometry (not a valid push direction).")
                continue

            action = game.encode_action(*move)

            if valid[action] == 0:
                print("Illegal move (violates rules).")
                continue

            return action

        except (ValueError, IndexError):
            print("Invalid input. Format: r c side (Example: 0 0 R) or 'q'")


def play_game():
    game = QuixoFast()
    state = game.get_initial_state()
    player = 1  # X starts
    move_count = 0
    while True:
        print_board(state)
        print(f"Player {'X' if player == 1 else 'O'}")

        canonical = game.change_perspective(state, player)

        # Get the action and check if the player chose to quit
        action = get_human_move(game, canonical)
        
        if action is None:
            print("Game terminated by user.")
            break

        state = game.get_next_state(state, action, player)
        move_count += 1
        value, done = game.get_value_and_terminated(state, player, move_count)

        if done:
            print_board(state)
            if value == 1:
                print(f"Player {'X' if player == 1 else 'O'} wins!")
            else:
                print(f"Player {'X' if player == 1 else 'O'} loses!")
            break

        player = -player

# play_game()

In [45]:
class ResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden, device):
        super().__init__()
        
        self.device = device
        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )
        
        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for i in range(num_resBlocks)]
        )
        
        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * game.size * game.size, game.action_size)
        )
        
        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.size * game.size, 1),
            nn.Tanh()
        )
        
        self.to(device)
        
    def forward(self, x):
        x = self.startBlock(x)
        for resBlock in self.backBone:
            x = resBlock(x)
        policy = self.policyHead(x)
        value = self.valueHead(x)
        return policy, value
        
        
class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)
        
    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x
        

In [46]:
class Node:
    def __init__(self, game, args, state, parent=None, action_taken=None, prior=0, visit_count=0, move_count=0):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken
        self.prior = prior
        self.move_count = move_count
        
        self.children = []
        
        self.visit_count = visit_count
        self.value_sum = 0
        
    def is_fully_expanded(self):
        return len(self.children) > 0
    
    def select(self):
        best_child = None
        best_ucb = -np.inf
        
        for child in self.children:
            ucb = self.get_ucb(child)
            if ucb > best_ucb:
                best_child = child
                best_ucb = ucb
                
        return best_child
    
    def get_ucb(self, child):
        if child.visit_count == 0:
            q_value = 0
        else:
            q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
        return q_value + self.args['C'] * (math.sqrt(self.visit_count) / (child.visit_count + 1)) * child.prior
    
    def expand(self, policy):
        for action, prob in enumerate(policy):
            if prob > 0:
                child_state = self.state.copy()
                child_state = self.game.get_next_state(child_state, action, 1)
                child_state = self.game.change_perspective(child_state, player=-1)

                child = Node(
                    self.game,
                    self.args,
                    child_state,
                    self,
                    action,
                    prob,
                    move_count=self.move_count + 1,
                )
                self.children.append(child)
                
        return child
            
    def backpropagate(self, value):
        self.value_sum += value
        self.visit_count += 1
        
        value = self.game.get_opponent_value(value)
        if self.parent is not None:
            self.parent.backpropagate(value)  


class MCTS:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model
        
    @torch.no_grad()
    def search(self, state):
        root = Node(self.game, self.args, state, visit_count=1, move_count=0)
        
        policy, _ = self.model(
            torch.tensor(self.game.get_encoded_state(state), device=self.model.device).unsqueeze(0)
        )
        policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()
        policy = (1 - self.args['dirichlet_epsilon']) * policy + self.args['dirichlet_epsilon'] \
            * np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size)
        
        valid_moves = self.game.get_valid_moves(state)
        policy *= valid_moves
        policy /= np.sum(policy)
        root.expand(policy)
        
        for search in range(self.args['num_searches']):
            node = root
            
            while node.is_fully_expanded():
                node = node.select()
                
            value, is_terminal = self.game.get_value_and_terminated(node.state, 1, move_count=node.move_count)
            value = self.game.get_opponent_value(value)
            
            if not is_terminal:
                policy, value = self.model(
                    torch.tensor(self.game.get_encoded_state(node.state), device=self.model.device).unsqueeze(0)
                )
                policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()
                valid_moves = self.game.get_valid_moves(node.state)
                policy *= valid_moves
                policy /= np.sum(policy)
                
                value = value.item()
                
                node.expand(policy)
                
            node.backpropagate(value)    
            
            
        action_probs = np.zeros(self.game.action_size)
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        return action_probs
        

In [47]:
class AlphaZero:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTS(game, args, model)
        
    def selfPlay(self):
        memory = []
        player = 1
        state = self.game.get_initial_state()
        move_count = 0
        
        while True:
            neutral_state = self.game.change_perspective(state, player)
            action_probs = self.mcts.search(neutral_state)
            
            memory.append((neutral_state, action_probs, player))
            
            eps = 1e-8
            temperature_action_probs = action_probs ** (1 / self.args['temperature'])
            temperature_action_probs = temperature_action_probs / (temperature_action_probs.sum() + eps)            
            action = np.random.choice(self.game.action_size, p=temperature_action_probs)
						
            state = self.game.get_next_state(state, action, player)
            move_count += 1
            
            value, is_terminal = self.game.get_value_and_terminated(state, player, move_count=move_count)
            
            if is_terminal:
                returnMemory = []
                for hist_neutral_state, hist_action_probs, hist_player in memory:
                    hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                    returnMemory.append((
                        self.game.get_encoded_state(hist_neutral_state),
                        hist_action_probs,
                        hist_outcome
                    ))
                return returnMemory
            
            player = self.game.get_opponent(player)
                
    def train(self, memory):
        random.shuffle(memory)
        for batchIdx in range(0, len(memory), self.args['batch_size']):
            sample = memory[batchIdx:min(len(memory) - 1, batchIdx + self.args['batch_size'])] # Change to memory[batchIdx:batchIdx+self.args['batch_size']] in case of an error
            state, policy_targets, value_targets = zip(*sample)
            
            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)
            
            state = torch.tensor(state, dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)
            
            out_policy, out_value = self.model(state)
            
            policy_loss = F.cross_entropy(out_policy, policy_targets)
            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
    
    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory = []
            
            self.model.eval()
            for selfPlay_iteration in trange(self.args['num_selfPlay_iterations']):
                memory += self.selfPlay()
                
            self.model.train()
            for epoch in trange(self.args['num_epochs']):
                self.train(memory)
            
            torch.save(self.model.state_dict(), f"model_{iteration}_{self.game}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{iteration}_{self.game}.pt")

In [48]:
class MCTSParallel:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model
        
    @torch.no_grad()
    def search(self, states, spGames):
        policy, _ = self.model(
            torch.tensor(self.game.get_encoded_state(states), device=self.model.device)
        )
        policy = torch.softmax(policy, axis=1).cpu().numpy()
        policy = (1 - self.args['dirichlet_epsilon']) * policy + self.args['dirichlet_epsilon'] \
            * np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size, size=policy.shape[0])
        
        for i, spg in enumerate(spGames):
            spg_policy = policy[i]
            valid_moves = self.game.get_valid_moves(states[i])
            spg_policy *= valid_moves
            spg_policy /= np.sum(spg_policy)

            spg.root = Node(self.game, self.args, states[i], visit_count=1, move_count=spg.move_count)
            spg.root.expand(spg_policy)
        
        for search in range(self.args['num_searches']):
            for spg in spGames:
                spg.node = None
                node = spg.root

                while node.is_fully_expanded():
                    node = node.select()

                value, is_terminal = self.game.get_value_and_terminated(node.state, 1, move_count=node.move_count)
                value = self.game.get_opponent_value(value)
                
                if is_terminal:
                    node.backpropagate(value)
                    
                else:
                    spg.node = node
                    
            expandable_spGames = [mappingIdx for mappingIdx in range(len(spGames)) if spGames[mappingIdx].node is not None]
                    
            if len(expandable_spGames) > 0:
                states = np.stack([spGames[mappingIdx].node.state for mappingIdx in expandable_spGames])
                
                policy, value = self.model(
                    torch.tensor(self.game.get_encoded_state(states), device=self.model.device)
                )
                policy = torch.softmax(policy, axis=1).cpu().numpy()
                value = value.cpu().numpy()
                
            for i, mappingIdx in enumerate(expandable_spGames):
                node = spGames[mappingIdx].node
                spg_policy, spg_value = policy[i], value[i]
                
                valid_moves = self.game.get_valid_moves(node.state)
                spg_policy *= valid_moves
                spg_policy /= np.sum(spg_policy)

                node.expand(spg_policy)
                node.backpropagate(spg_value)

In [ ]:
class AlphaZeroParallel:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTSParallel(game, args, model)
        
    def selfPlay(self):
        return_memory = []
        player = 1
        # Initialize parallel games
        spGames = [SPG(self.game) for _ in range(self.args['num_parallel_games'])]
        
        while len(spGames) > 0:
            # 1. Vectorized Perspective Change
            # We stack all current states into one (Batch, 5, 5) array
            states = np.stack([spg.state for spg in spGames])
            
            # The new Quixo class handles this multiplication across the whole batch at once
            neutral_states = self.game.change_perspective(states, player)
            
            # 2. MCTS Search (remains largely the same, but now fed faster states)
            self.mcts.search(neutral_states, spGames)
            
            # Iterate backwards through games to allow safe deletion
            for i in range(len(spGames))[::-1]:
                spg = spGames[i]
                
                # Calculate policy from MCTS visit counts
                action_probs = np.zeros(self.game.action_size)
                for child in spg.root.children:
                    action_probs[child.action_taken] = child.visit_count
                
                sum_visits = np.sum(action_probs)
                if sum_visits > 0:
                    action_probs /= sum_visits
                else:
                    # Fallback if MCTS failed to expand (rare)
                    action_probs = self.game.get_valid_moves(spg.state).astype(float)
                    action_probs /= action_probs.sum()

                # Store the state in memory (ensure it's the neutral/canonical state)
                # We use .copy() to prevent reference issues during training
                canonical_state = self.game.change_perspective(spg.state, player)
                spg.memory.append((canonical_state, action_probs, player))

                # Temperature-based action selection
                temp = self.args.get('temperature', 1.0)
                if temp == 0:
                    action = np.argmax(action_probs)
                else:
                    lp = np.power(action_probs, 1 / temp)
                    action = np.random.choice(self.game.action_size, p=lp / np.sum(lp))

                # 3. Optimized State Transition
                # Uses the Numba-accelerated fast_push under the hood
                spg.state = self.game.get_next_state(spg.state, action, player)
                spg.move_count += 1

                # 4. Vectorized Terminal Check
                # Now passing move_count to trigger the 100-move limit
                value, is_terminal = self.game.get_value_and_terminated(
                    spg.state, player, move_count=spg.move_count
                )

                if is_terminal:
                    for hist_neutral_state, hist_action_probs, hist_player in spg.memory:
                        # Determine outcome for historical player
                        hist_outcome = value if hist_player == player else -value
                        
                        return_memory.append((
                            self.game.get_encoded_state(hist_neutral_state),
                            hist_action_probs,
                            hist_outcome
                        ))
                    del spGames[i]
            
            # Switch player
            player = -player
            
        return return_memory

    # ... train and learn methods remain the same ...
                
    def train(self, memory):
        random.shuffle(memory)
        for batchIdx in range(0, len(memory), self.args['batch_size']):
            sample = memory[batchIdx:min(len(memory) - 1, batchIdx + self.args['batch_size'])] # Change to memory[batchIdx:batchIdx+self.args['batch_size']] in case of an error
            state, policy_targets, value_targets = zip(*sample)
            
            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)
            
            state = torch.tensor(state, dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)
            
            out_policy, out_value = self.model(state)
            
            policy_loss = F.cross_entropy(out_policy, policy_targets)
            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()


    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory = []
            
            self.model.eval()
            start_time = time.time()
            total_states = 0
            
            # We calculate how many loops of parallel games we need
            num_loops = self.args['num_selfPlay_iterations'] // self.args['num_parallel_games']
            
            print(f"--- Iteration {iteration} Self-Play ---")
            for selfPlay_iteration in trange(num_loops):
                print(f"Self-Play Loop {selfPlay_iteration + 1}/{num_loops}")
                batch_memory = self.selfPlay()
                memory += batch_memory
                total_states += len(batch_memory)
                
            end_time = time.time()
            elapsed = end_time - start_time
            gps = self.args['num_selfPlay_iterations'] / elapsed
            sps = total_states / elapsed
            
            print(f"Performance Metrics:")
            print(f" - Total Time: {elapsed:.2f}s")
            print(f" - Games per Second: {gps:.2f}")
            print(f" - States per Second: {sps:.2f}")
            
            self.model.train()
            print(f"--- Iteration {iteration} Training ---")
            for epoch in trange(self.args['num_epochs']):
                print(f"Epoch {epoch + 1}/{self.args['num_epochs']}")
                self.train(memory)
            
            print(f"Saving model and optimizer for iteration {iteration}...")
            
            timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

            torch.save(
                self.model.state_dict(),
                f"model_fast_{timestamp}_{iteration}_{self.game}.pt"
            )
            torch.save(
                self.optimizer.state_dict(),
                f"optimizer_fast_{timestamp}_{iteration}_{self.game}.pt"
            )    
class SPG:
    def __init__(self, game):
        self.state = game.get_initial_state()
        self.memory = []
        self.root = None
        self.node = None
        self.move_count = 0

In [ ]:
game = QuixoFast()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

model = ResNet(game, 9, 128, device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)

args = {
    'C': 2,
    'num_searches': 200, # 600
    'num_iterations': 8, # 8
    'num_selfPlay_iterations': 512, # 500
    'num_parallel_games': 128, # 100
    'num_epochs': 4, # 4
    'batch_size': 128,
    'temperature': 1.25,
    'dirichlet_epsilon': 0.25,
    'dirichlet_alpha': 0.3
}

alphaZero = AlphaZeroParallel(model, optimizer, game, args)
alphaZero.learn()

Using device: mps
--- Iteration 0 Self-Play ---


  0%|          | 0/4 [00:00<?, ?it/s]

Self-Play Loop 1/4
Self-Play Loop 2/4


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x104291c10>>
Traceback (most recent call last):
  File "/Users/danielgagliardi/Library/Python/3.12/lib/python/site-packages/ipykernel/ipkernel.py", line 797, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [21]:
torch.save(model.state_dict(), f"model_test_Q.pt")

In [ ]:
import numpy as np
import torch

def print_board(state):
    symbols = {0: ".", 1: "X", -1: "O"}
    print()
    for r in range(state.shape[0]):
        print(" ".join(symbols[x] for x in state[r]))
    print()


def get_human_action(game, state):
    side_map = {
        "L": lambda r, c: (r, c, r, 0),
        "R": lambda r, c: (r, c, r, game.size - 1),
        "T": lambda r, c: (r, c, 0, c),
        "B": lambda r, c: (r, c, game.size - 1, c),
    }

    valid = game.get_valid_moves(state)

    while True:
        try:
            raw = input("Enter move (r c side[L/R/T/B]): ").strip().split()
            r, c = int(raw[0]), int(raw[1])
            side = raw[2].upper()

            if side not in side_map:
                raise ValueError

            move = side_map[side](r, c)

            if move not in game.action_map:
                print("Invalid geometry")
                continue

            action = game.encode_action(*move)

            if valid[action] == 0:
                print("Illegal move")
                continue

            return action

        except Exception:
            print("Invalid input format")


# --- Setup ---
game = QuixoFast()
player = 1

args = {
    'C': 2,
    'num_searches': 600,
    'dirichlet_epsilon': 0.,
    'dirichlet_alpha': 0.3
}

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = ResNet(game, 9, 128, device)
model.load_state_dict(torch.load("model_7_Quixo.pt", map_location=device))
model.eval()

mcts = MCTS(game, args, model)

state = game.get_initial_state()


# --- Game Loop ---
while True:
    print_board(state)

    if player == 1:
        print("Player X (human)")
        canonical = game.change_perspective(state, player)
        action = get_human_action(game, canonical)

    else:
        print("Player O (AI)")
        neutral_state = game.change_perspective(state, player)
        mcts_probs = mcts.search(neutral_state)
        action = np.argmax(mcts_probs)

    state = game.get_next_state(state, action, player)

    value, is_terminal = game.get_value_and_terminated(state, player)

    if is_terminal:
        print_board(state)

        if value == 1:
            print(f"Player {'X' if player == 1 else 'O'} wins")
        else:
            print(f"Player {'X' if player == 1 else 'O'} loses")

        break

    player = game.get_opponent(player)


. . . . .
. . . . .
. . . . .
. . . . .
. . . . .

Player X (human)

. . . . .
. . . . .
. . . . .
. . . . .
X . . . .

Player O (AI)

O . . . .
. . . . .
. . . . .
. . . . .
X . . . .

Player X (human)

O . . . .
. . . . .
. . . . .
. . . . .
X . . . X

Player O (AI)

O . . . .
. . . . .
O . . . .
. . . . .
X . . . X

Player X (human)

O . . . .
. . . . .
O . . . .
. . . . X
X . . . X

Player O (AI)

O . . . .
. . . . .
O . . . .
. . . . X
X . O . X

Player X (human)

O . . . .
. . . . .
O . . . X
. . . . X
X . O . X

Player O (AI)

O . . . .
. . . . .
O . . . X
. . . . X
O X . O X

Player X (human)

O . . . .
. . . . X
O . . . X
. . . . X
O X . O X

Player O (AI)

O . . . .
. . . . X
O . . . X
. X . . X
O O . O X

Player X (human)

O . . . X
. . . . X
O . . . X
. X . . X
O O . O X

Player X wins
